# Merge modelling panel

Aggregate interim vector / virus tables into supervised spatial–temporal units (spec `04`).

| Item | Detail |
|------|--------|
| Inputs | `data/02_interim/vectors_normalized.parquet`, `viruses_normalized.parquet` |
| Outputs | `data/03_processed/modelling_panel.parquet` + `modelling_panel_manifest.json` |
| Tier 1 | `(h3_cell, year, month, vector_group)` when VectorNet has H3 |
| Tier 2 | `(country_iso2, year, month, vector_group)` when VectorNet lacks H3 |
| Training filter | **VectorNet coverage required** — virus-only units excluded from this panel |
| Ecology | Join only within the same `vector_group` |

Country-only virus rows attach at Tier 2 only (no invented H3).

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_processed_dir,
    default_vector_raw_dir,
    default_virus_raw_dir,
    find_repo_root,
)
from vector_guard.resolve import (
    GEO_PRECISION_CENTROID,
    GEO_PRECISION_EXACT,
    H3_RESOLUTION,
    load_interim_vectors,
    load_interim_viruses,
)
from vector_guard.virus_extract import INSDC_SOURCES, OUTBREAK_SOURCES

REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PANEL_PATH = PROCESSED_DIR / "modelling_panel.parquet"
MANIFEST_PATH = PROCESSED_DIR / "modelling_panel_manifest.json"

REPO_ROOT, PANEL_PATH, MANIFEST_PATH, H3_RESOLUTION

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/03_processed/modelling_panel.parquet'),
 PosixPath('/workspaces/vector-guard/data/03_processed/modelling_panel_manifest.json'),
 4)

## Load interim sides

In [2]:
vectors = load_interim_vectors()
viruses = load_interim_viruses()

{
    "vectors": len(vectors),
    "viruses": len(viruses),
    "H3_RESOLUTION": H3_RESOLUTION,
}

{'vectors': 120158, 'viruses': 168277, 'H3_RESOLUTION': 4}

## Filter to panel-key-ready rows

Require `vector_group`, `year`, and `month`. Spatial key must be H3 and/or ISO-2 depending on tier.

In [3]:
def _ready(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[
        (df["vector_group"].fillna("") != "")
        & df["year"].notna()
        & df["month"].notna()
    ].copy()


vectors_r = _ready(vectors)
viruses_r = _ready(viruses)

vectors_r["year"] = vectors_r["year"].astype(int)
vectors_r["month"] = vectors_r["month"].astype(int)
viruses_r["year"] = viruses_r["year"].astype(int)
viruses_r["month"] = viruses_r["month"].astype(int)

vectors_t1 = vectors_r.loc[vectors_r["h3_cell"].notna()].copy()
vectors_t2 = vectors_r.loc[
    vectors_r["h3_cell"].isna() & (vectors_r["country_iso2"].fillna("") != "")
].copy()

viruses_t1 = viruses_r.loc[viruses_r["h3_cell"].notna()].copy()
viruses_t2 = viruses_r.loc[
    viruses_r["h3_cell"].isna() & (viruses_r["country_iso2"].fillna("") != "")
].copy()

{
    "vectors_ready": len(vectors_r),
    "vectors_tier1_rows": len(vectors_t1),
    "vectors_tier2_rows": len(vectors_t2),
    "viruses_ready": len(viruses_r),
    "viruses_tier1_rows": len(viruses_t1),
    "viruses_tier2_rows": len(viruses_t2),
}

{'vectors_ready': 119165,
 'vectors_tier1_rows': 119165,
 'vectors_tier2_rows': 0,
 'viruses_ready': 110239,
 'viruses_tier1_rows': 64276,
 'viruses_tier2_rows': 45647}

## Aggregate helpers

In [4]:
def _parse_individual_count(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def aggregate_vectors_tier1(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_ic"] = _parse_individual_count(work["individual_count"])
    keys = ["h3_cell", "year", "month", "vector_group"]
    grouped = work.groupby(keys, dropna=False)
    out = grouped.agg(
        vector_detection_count=("record_id", "size"),
        individual_count_sum=("_ic", "sum"),
        individual_count_mean=("_ic", "mean"),
        country_iso2=(
            "country_iso2",
            lambda s: s.fillna("").replace("", pd.NA).mode(dropna=True).iloc[0]
            if s.fillna("").replace("", pd.NA).dropna().size
            else "",
        ),
    ).reset_index()
    out["spatial_tier"] = "h3"
    out["vector_present"] = 1
    out["individual_count_sum"] = out["individual_count_sum"].fillna(0.0)
    return out


def aggregate_vectors_tier2(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_ic"] = _parse_individual_count(work["individual_count"])
    keys = ["country_iso2", "year", "month", "vector_group"]
    grouped = work.groupby(keys, dropna=False)
    out = grouped.agg(
        vector_detection_count=("record_id", "size"),
        individual_count_sum=("_ic", "sum"),
        individual_count_mean=("_ic", "mean"),
    ).reset_index()
    out["spatial_tier"] = "country"
    out["h3_cell"] = pd.NA
    out["vector_present"] = 1
    out["individual_count_sum"] = out["individual_count_sum"].fillna(0.0)
    return out


def aggregate_viruses_tier1(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_insdc"] = work["source"].isin(INSDC_SOURCES).astype(int)
    work["_outbreak"] = work["source"].isin(OUTBREAK_SOURCES).astype(int)
    work["_exact"] = (work["geo_precision"] == GEO_PRECISION_EXACT).astype(float)
    work["_centroid"] = (work["geo_precision"] == GEO_PRECISION_CENTROID).astype(float)
    keys = ["h3_cell", "year", "month", "vector_group"]
    out = (
        work.groupby(keys, dropna=False)
        .agg(
            virus_detection_count=("record_id", "size"),
            virus_source_n_sequence=("_insdc", "sum"),
            virus_source_n_outbreak=("_outbreak", "sum"),
            share_virus_exact_coords=("_exact", "mean"),
            share_virus_subnational_centroid=("_centroid", "mean"),
        )
        .reset_index()
    )
    out["virus_detected"] = (out["virus_detection_count"] > 0).astype(int)
    return out


def aggregate_viruses_tier2(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_insdc"] = work["source"].isin(INSDC_SOURCES).astype(int)
    work["_outbreak"] = work["source"].isin(OUTBREAK_SOURCES).astype(int)
    keys = ["country_iso2", "year", "month", "vector_group"]
    out = (
        work.groupby(keys, dropna=False)
        .agg(
            virus_detection_count=("record_id", "size"),
            virus_source_n_sequence=("_insdc", "sum"),
            virus_source_n_outbreak=("_outbreak", "sum"),
        )
        .reset_index()
    )
    out["virus_detected"] = (out["virus_detection_count"] > 0).astype(int)
    out["share_virus_exact_coords"] = pd.NA
    out["share_virus_subnational_centroid"] = pd.NA
    return out

## Build Tier 1 + Tier 2 panels (VectorNet-left join)

In [5]:
vec_t1 = aggregate_vectors_tier1(vectors_t1)
vir_t1 = aggregate_viruses_tier1(viruses_t1)
panel_t1 = vec_t1.merge(
    vir_t1,
    on=["h3_cell", "year", "month", "vector_group"],
    how="left",
)

vec_t2 = aggregate_vectors_tier2(vectors_t2)
vir_t2 = aggregate_viruses_tier2(viruses_t2)
panel_t2 = vec_t2.merge(
    vir_t2,
    on=["country_iso2", "year", "month", "vector_group"],
    how="left",
)

panel = pd.concat([panel_t1, panel_t2], ignore_index=True, sort=False)

# Virus absences in vector-covered units → explicit zeros (not NA)
for col, fill in (
    ("virus_detection_count", 0),
    ("virus_detected", 0),
    ("virus_source_n_sequence", 0),
    ("virus_source_n_outbreak", 0),
):
    panel[col] = panel[col].fillna(fill).astype(int)

panel["vector_present"] = 1
panel["country_iso2"] = panel["country_iso2"].fillna("")

column_order = [
    "spatial_tier",
    "h3_cell",
    "country_iso2",
    "year",
    "month",
    "vector_group",
    "vector_present",
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "virus_detected",
    "virus_detection_count",
    "virus_source_n_sequence",
    "virus_source_n_outbreak",
    "share_virus_exact_coords",
    "share_virus_subnational_centroid",
]
panel = panel[column_order].sort_values(
    ["spatial_tier", "vector_group", "year", "month", "h3_cell", "country_iso2"],
    kind="mergesort",
).reset_index(drop=True)

display(panel["spatial_tier"].value_counts())
display(panel.groupby(["spatial_tier", "vector_group"]).size().rename("units").reset_index())
display(
    panel.groupby("spatial_tier")
    .agg(
        units=("vector_present", "size"),
        virus_positive=("virus_detected", "sum"),
        virus_positive_rate=("virus_detected", "mean"),
    )
)
panel.head()

spatial_tier
h3    12995
Name: count, dtype: int64

,spatial_tier,vector_group,units
0,h3,biting_midge,5038
1,h3,mosquito,6674
2,h3,sandfly,425
3,h3,tick,858


,units,virus_positive,virus_positive_rate
spatial_tier,,,
h3,12995,45,0.003463


,spatial_tier,h3_cell,country_iso2,year,month,vector_group,vector_present,vector_detection_count,individual_count_sum,individual_count_mean,virus_detected,virus_detection_count,virus_source_n_sequence,virus_source_n_outbreak,share_virus_exact_coords,share_virus_subnational_centroid
0,h3,8419519ffffffff,GB,1974,3,biting_midge,1,14,11,0.785714,0,0,0,0,NaN,NaN
1,h3,84195b7ffffffff,GB,1974,3,biting_midge,1,14,38,2.714286,0,0,0,0,NaN,NaN
2,h3,8419519ffffffff,GB,1974,4,biting_midge,1,14,15,1.071429,0,0,0,0,NaN,NaN
3,h3,84195b7ffffffff,GB,1974,4,biting_midge,1,21,9,0.428571,0,0,0,0,NaN,NaN
4,h3,8419519ffffffff,GB,1974,5,biting_midge,1,14,1,0.071429,0,0,0,0,NaN,NaN


## Manifest + write

In [6]:
def _sha256(path: Path) -> str | None:
    if not path.exists():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


virus_manifest = default_virus_raw_dir(REPO_ROOT) / "manifest.json"
vector_manifest = default_vector_raw_dir(REPO_ROOT) / "manifest.json"

assert panel["vector_present"].eq(1).all()
assert set(panel["spatial_tier"].unique()) <= {"h3", "country"}
assert panel.loc[panel["spatial_tier"] == "h3", "h3_cell"].notna().all()
assert panel.loc[panel["spatial_tier"] == "country", "h3_cell"].isna().all()
assert panel.loc[panel["spatial_tier"] == "country", "country_iso2"].ne("").all()
assert panel["month"].between(1, 12).all()
assert panel["virus_detected"].isin([0, 1]).all()

tier_counts = panel["spatial_tier"].value_counts().to_dict()
date_min = f"{int(panel['year'].min())}-{int(panel.loc[panel['year'].idxmin(), 'month']):02d}"
date_max = f"{int(panel['year'].max())}-{int(panel.loc[panel['year'].idxmax(), 'month']):02d}"
# Prefer true min/max year-month pairs
ym = panel.assign(_ym=panel["year"] * 100 + panel["month"])
ymin_row = ym.loc[ym["_ym"].idxmin()]
ymax_row = ym.loc[ym["_ym"].idxmax()]
date_min = f"{int(ymin_row['year'])}-{int(ymin_row['month']):02d}"
date_max = f"{int(ymax_row['year'])}-{int(ymax_row['month']):02d}"

virus_geo_mix = {
    "tier1_units_with_virus": int(
        ((panel["spatial_tier"] == "h3") & (panel["virus_detected"] == 1)).sum()
    ),
    "mean_share_exact_coords_on_positive_tier1": float(
        panel.loc[
            (panel["spatial_tier"] == "h3") & (panel["virus_detected"] == 1),
            "share_virus_exact_coords",
        ]
        .dropna()
        .mean()
    )
    if ((panel["spatial_tier"] == "h3") & (panel["virus_detected"] == 1)).any()
    else None,
    "mean_share_centroid_on_positive_tier1": float(
        panel.loc[
            (panel["spatial_tier"] == "h3") & (panel["virus_detected"] == 1),
            "share_virus_subnational_centroid",
        ]
        .dropna()
        .mean()
    )
    if ((panel["spatial_tier"] == "h3") & (panel["virus_detected"] == 1)).any()
    else None,
}

manifest = {
    "H3_RESOLUTION": H3_RESOLUTION,
    "panel_rows": int(len(panel)),
    "rows_by_spatial_tier": {str(k): int(v) for k, v in tier_counts.items()},
    "virus_positive_units": int(panel["virus_detected"].sum()),
    "virus_positive_rate": float(panel["virus_detected"].mean()),
    "date_range": {"min": date_min, "max": date_max},
    "vector_groups": sorted(panel["vector_group"].unique().tolist()),
    "training_filter": (
        "Supervised panel includes only units with VectorNet coverage; "
        "virus-only units are excluded. Virus zeros mean no reported detection "
        "in the merged feeds for that unit-window."
    ),
    "temporal_grain": "year x month",
    "tier1_key": ["h3_cell", "year", "month", "vector_group"],
    "tier2_key": ["country_iso2", "year", "month", "vector_group"],
    "virus_geo_precision_mix": virus_geo_mix,
    "source_manifest_sha256": {
        "virus_manifest.json": _sha256(virus_manifest),
        "vector_manifest.json": _sha256(vector_manifest),
    },
    "inputs": {
        "vectors_normalized": "data/02_interim/vectors_normalized.parquet",
        "viruses_normalized": "data/02_interim/viruses_normalized.parquet",
    },
    "outputs": {
        "panel": str(PANEL_PATH.relative_to(REPO_ROOT)),
        "manifest": str(MANIFEST_PATH.relative_to(REPO_ROOT)),
    },
}

panel.to_parquet(PANEL_PATH, index=False)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print(f"Wrote {PANEL_PATH} ({len(panel):,} rows)")
print(f"Wrote {MANIFEST_PATH}")
display(pd.Series({k: manifest[k] for k in ("H3_RESOLUTION", "panel_rows", "virus_positive_units", "virus_positive_rate", "date_range")}))
display(panel.head(12))
display(panel.tail(5))

Wrote /workspaces/vector-guard/data/03_processed/modelling_panel.parquet (12,995 rows)
Wrote /workspaces/vector-guard/data/03_processed/modelling_panel_manifest.json


H3_RESOLUTION                                              4
panel_rows                                             12995
virus_positive_units                                      45
virus_positive_rate                                 0.003463
date_range              {'min': '1957-09', 'max': '2025-04'}
dtype: object

,spatial_tier,h3_cell,country_iso2,year,month,vector_group,vector_present,vector_detection_count,individual_count_sum,individual_count_mean,virus_detected,virus_detection_count,virus_source_n_sequence,virus_source_n_outbreak,share_virus_exact_coords,share_virus_subnational_centroid
0,h3,8419519ffffffff,GB,1974,3,biting_midge,1,14,11,0.785714,0,0,0,0,NaN,NaN
1,h3,84195b7ffffffff,GB,1974,3,biting_midge,1,14,38,2.714286,0,0,0,0,NaN,NaN
2,h3,8419519ffffffff,GB,1974,4,biting_midge,1,14,15,1.071429,0,0,0,0,NaN,NaN
3,h3,84195b7ffffffff,GB,1974,4,biting_midge,1,21,9,0.428571,0,0,0,0,NaN,NaN
4,h3,8419519ffffffff,GB,1974,5,biting_midge,1,14,1,0.071429,0,0,0,0,NaN,NaN
5,h3,84195b7ffffffff,GB,1974,5,biting_midge,1,7,4,0.571429,0,0,0,0,NaN,NaN
6,h3,8419519ffffffff,GB,1974,6,biting_midge,1,7,0,0.000000,0,0,0,0,NaN,NaN
7,h3,84195b7ffffffff,GB,1974,6,biting_midge,1,14,4,0.285714,0,0,0,0,NaN,NaN
8,h3,8419519ffffffff,GB,1974,7,biting_midge,1,14,6,0.428571,0,0,0,0,NaN,NaN
9,h3,84195b7ffffffff,GB,1974,7,biting_midge,1,21,4,0.190476,0,0,0,0,NaN,NaN


,spatial_tier,h3_cell,country_iso2,year,month,vector_group,vector_present,vector_detection_count,individual_count_sum,individual_count_mean,virus_detected,virus_detection_count,virus_source_n_sequence,virus_source_n_outbreak,share_virus_exact_coords,share_virus_subnational_centroid
12990,h3,841fb29ffffffff,FR,2022,6,tick,1,9,27,3.000000,0,0,0,0,NaN,NaN
12991,h3,841e037ffffffff,HU,2022,7,tick,1,4,4714,1178.500000,0,0,0,0,NaN,NaN
12992,h3,841fb29ffffffff,FR,2022,7,tick,1,9,13,1.444444,0,0,0,0,NaN,NaN
12993,h3,841fb29ffffffff,FR,2022,8,tick,1,9,5,0.555556,0,0,0,0,NaN,NaN
12994,h3,841fb29ffffffff,FR,2022,12,tick,1,9,34,3.777778,0,0,0,0,NaN,NaN
